# 🛠️ Mərhələ 1: Datanın Yüklənməsi və İlkin Struktur Analizi

Bu bölmədə layihədə istifadə olunacaq Kredit Risk datasetini Python mühitinə yükləyirik,
verilənlərin ümumi mənzərəsini vizuallaşdırırıq və verilənlər bazasındakı sütunların tiplərini analiz edirik.
Məqsədimiz hər bir sütunun biznes mənası ilə tanış olmaqdır.

In [14]:
import pandas as pd
import numpy as np

# Datanı layihə strukturuna uyğun olaraq raw qovluğundan oxuduruq
df = pd.read_csv('../data/raw/credit_risk_dataset.csv')

# Datanın ilk 5 sətirinə vizual baxış keçiririk
df.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


### 📊 Datanın Texniki Strukturunun (`df.info()`) Analizi

Datanın ümumi sətir sayını, sütun adlarını, proqramlaşdırma dillərindəki mövcud data tiplərini
(integer, float, object) və daxildəki boş (null) xanaları müəyyən etmək üçün `df.info()` metodundan istifadə edirik.
Bu metod bizə növbəti mərhələdə hansı sütunlar üzərində məlumat təmizlənməsi (Data Cleaning) edəcəyimizi göstərəcək.

In [15]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


### 📈 Rəqəmsal Sütunların İlkin Statistik Analizi (`df.describe()`)

Dataset daxilindəki bütün rəqəmsal sütunların riyazi və statistik xülasəsini
(orta qiymət, minimum, maksimum, median) görmək üçün `df.describe()` metodunu işlədirik.
Bu analiz vasitəsilə biz datada mövcud ola biləcək məntiqsiz və ekstremal kənarlaşmaları
(Outliers - məsələn, insan yaşı üçün real olmayan rəqəmlər) ilkin olaraq müəyyənləşdiririk.

In [16]:
# Rəqəmsal məlumatların statistik xülasəsi
df.describe()

,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_cred_hist_length
count,32581.000000,3.258100e+04,31686.000000,32581.000000,29465.000000,32581.000000,32581.000000,32581.000000
mean,27.734600,6.607485e+04,4.789686,9589.371106,11.011695,0.218164,0.170203,5.804211
std,6.348078,6.198312e+04,4.142630,6322.086646,3.240459,0.413006,0.106782,4.055001
min,20.000000,4.000000e+03,0.000000,500.000000,5.420000,0.000000,0.000000,2.000000
25%,23.000000,3.850000e+04,2.000000,5000.000000,7.900000,0.000000,0.090000,3.000000
50%,26.000000,5.500000e+04,4.000000,8000.000000,10.990000,0.000000,0.150000,4.000000
75%,30.000000,7.920000e+04,7.000000,12200.000000,13.470000,0.000000,0.230000,8.000000
max,144.000000,6.000000e+06,123.000000,35000.000000,23.220000,1.000000,0.830000,30.000000


# 🧹 Mərhələ 2: Data Cleaning (Məlumatların Təmizlənməsi)

Bu mərhələdə datanın keyfiyyətini artırmaq üçün eksik (null) dəyərləri analiz edəcəyik 
və datadakı məntiqsiz kənarlaşmaları (Outliers) təmizləyəcəyik. 
Təmiz data analizin dəqiqliyi üçün ən vacib şərtdir.

### 1. Boş (Null) Xanaların Tapılması
    

In [18]:
# Hansı sütunda neçə dənə boşluq olduğunu dəqiq sayırıq
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0
dtype: int64

### 2. Boş Xanaların Doldurulması (Imputation)
Yuxarıdan da göründüyü kimi cəmi 2 sütunda null dəyərlər var.person_emp_length və loan_int_rate.
Analiz zamanı datanı itirməmək üçün boş xanaları silmirik, onları median (orta göstərici) ilə doldururuq. 
- `person_emp_length` (Müştərinin iş ili) sütunundakı boşluqları median ilə doldururuq.
- `loan_int_rate` (Kredit faiz dərəcəsi) sütunundakı boşluqları median ilə doldururuq.

In [29]:
# İş ili sütunundakı boşluqları median ilə doldururuq
df['person_emp_length'] = df['person_emp_length'].fillna(df['person_emp_length'].median())

# Kredit faizi sütunundakı boşluqları median ilə doldururuq
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].median())

# Yoxlayırıq ki, boş xana qaldımı
df.isnull().sum()

person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
person_emp_lenght             0
dtype: int64

### 3. Anomaliyaların (Outliers) Təmizlənməsi
`df.describe()` metodunda gördüyümüz kimi, `person_age` (yaş) sütununda 100-dən böyük (məsələn, 123 və ya 144) qeyri-real insan yaşları mövcuddur. 
Bank sektorunda bu cür məlumatlar sistem xətası hesab olunur. Yaşı 100-dən kiçik olan məntiqi dataları saxlayırıq, qalanlarını filterləyib silirik.

In [30]:
# Yaşı 100-dən çox olan sətirlərin sayına baxırıq
outliers_count = df[df['person_age'] > 100].shape[0]
print(f"Datadakı yaş anomaliyalarının sayı: {outliers_count}")

# Yalnız yaşı 100 və 100-dən aşağı olan müştəriləri saxlayırıq
df = df[df['person_age'] <= 100]

# Yoxlamaq üçün yenidən yaş sütununun maksimum dəyərinə baxırıq
print(f"Təmizlənmədən sonra maksimum yaş: {df['person_age'].max()}")

Datadakı yaş anomaliyalarının sayı: 5
Təmizlənmədən sonra maksimum yaş: 94


### İş Stajı Anomaliyasının (`person_emp_length`) Təmizlənməsi

Datanı incələyərkən məlum oldu ki, `person_emp_length` (müştərinin iş ili) sütununda 123 il kimi qeyri-real və məntiqsiz bir maksimum dəyər mövcuddur.
İnsan ömrü və iş stajı reallıqlarını nəzərə alaraq, 60 ildən çox iş stajı göstərilən qeydləri (sistem xətalarını) datadan filterləyib kənara qoyuruq.

In [33]:
# İş ili 60-dan çox olan məntiqsiz sətirlərin sayına baxırıq
emp_outliers = df[df['person_emp_length'] > 60].shape[0]
print(f"Datadakı iş stajı anomaliyalarının sayısı: {emp_outliers}")

# Yalnız iş stajı 60 il və ondan aşağı olan insanları saxlayırıq
df = df[df['person_emp_length'] <= 60]

# Yoxlamaq üçün yenidən iş stajı sütununun maksimum dəyərinə baxırıq
print(f"Təmizlənmədən sonra maksimum iş stajı (il): {df['person_emp_length'].max()}")

Datadakı iş stajı anomaliyalarının sayısı: 2
Təmizlənmədən sonra maksimum iş stajı (il): 41.0


### 4. Təkrar Sətirlərin (Duplicates) Silinməsi
Eyni müştəri qeydinin datada dublikat (təkrar) olub-olmadığını yoxlayırıq və əgər varsa, onları datadan kənarlaşdırırıq.

In [35]:
# Təkrar sətirlərin sayını yoxlayırıq
print(f"Təkrar sətir sayı: {df.duplicated().sum()}")

# Təkrar sətirlər varsa silirik və indeksi yeniləyirik
df = df.drop_duplicates().reset_index(drop=True)

# Yekun datanın ölçüsünə (sətir və sütun sayına) baxırıq
print(f"Yekun təmiz data ölçüsü: {df.shape}")

Təkrar sətir sayı: 0
Yekun təmiz data ölçüsü: (32409, 13)


### 💾 5. Təmizlənmiş Datanın Yaddaşa Yazılması (Exporting Cleaned Data)

Data təmizlənməsi prosesi uğurla başa çatdı. 
İndi isə növbəti mərhələlərdə (EDA və Power BI vizuallaşdırmasında) istifadə etmək üçün 
bu yekun master-datanı layihə strukturumuza uyğun olaraq `data/cleaned/` qovluğuna eksport edirik.

In [36]:
# Təmiz datanı indeks sətirləri olmadan cleaned qovluğuna yazdırırıq
df.to_csv('../data/cleaned/credit_risk_cleaned.csv', index=False)

print("Təmiz data uğurla 'data/cleaned/credit_risk_cleaned.csv' ünvanına qeyd edildi!")

Təmiz data uğurla 'data/cleaned/credit_risk_cleaned.csv' ünvanına qeyd edildi!
